# Create an `Photom` Reference File using the `RFP`

This notebook will guide users through creating a CRDS format compliant `Photom` Reference file with the RFP. 

-----

## Setup

In order to run this notebook and generate the Mask reference file, a `conda` environment must be created that has the `roman-wfi-reference-file-pipeline` code installed. 

To download `conda` on your machine, please follow the installation instructions on the [`conda` website](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html). 

Once `conda` is installed, go to [the RFP's Github page](https://github.com/spacetelescope/roman-wfi-reference-pipeline) and follow [the RFP installation instructions](https://github.com/spacetelescope/roman-wfi-reference-pipeline#installation).

When the code is properly installed, users can run the notebook examples.

-----

## Imports

A few packages are needed when running this notebook. We will import them in the cell below:

In [ ]:
import glob
import os

import asdf

from wfi_reference_pipeline.reference_types.photom.photom import Photom, gain, pam
from wfi_reference_pipeline.resources.make_dev_meta import MakeDevMeta

## About the RFP `Photom` Module

The `Photom` module is designed to create Roman WFI photom reference file that provides information on the flux calibration. The module is consisted of 3 parts: the `Photom` class and two standalone functions called `gain` and `pam`. 

`gain` queries the CRDS server for the available gain reference files, stores them locally, computes the sigma clipped median value and the standard deviation, and builds the gain dictionary to be used by the `Photom` class. The gain reference files from CRDS are not IPC-corrected and therefore, the `gain` function handles the correction itself by dividing a correction factor (1.08) from the sigma clipped median gain values. 

`pam` uses an independent `PixelArea` module in the RFP. The `PixelArea` class within the module is called to compute the pixel area map of the science pixels (4088 x 4088). 

The `Photom` class inherits from the `ReferenceType` base class. There are no user-required inputs for the `Photom` object but the class requires the gain and the pixel area information for each detector. 

### Create photom reference files for each detector

The code snippet below shows how to generate CRDS-compatible photom reference files using the `Photom` class in the module. We will loop through all 18 detectors and generate 18 ASDF files, one for each detector. 

We will first create a `MakeDevMeta` object and update some metadata. These metadata will be used in the CRDS submission. 

In [ ]:
tmp = MakeDevMeta(ref_type='PHOTOM')
print("The default metadata values are: ", tmp.meta_photom)

# Manually setting various metadata
tmp.meta_photom.use_after = '2020-05-01T00:00:00.000'
tmp.meta_photom.author = "RFP"

print("The new metadata values are: ", tmp.meta_photom)

Now we can create the `Photom` object. We will set meta_data to `tmp.meta_photom`. We will also specify an output filename for the reference file (we will use `roman_photom_file_wfi##.asdf` where ## corresponds to the detector number) and we will set `clobber` to `True` so we can overwrite existing files.

Once we successfully create the `Photom` object, we build the photom dictionary using `make_photom()` and generate the reference files using the function `generate_outfile()`. We will loop through all 18 detectors and create 18 output files. 

In [ ]:
# Creating the Photom object
rfp_photom = Photom(meta_data=tmp.meta_photom, 
                     outfile=f'roman_photom_file_wfi{tmp.meta_photom.instrument_detector:02}.asdf',
                     clobber=True)

# Loop through all 18 detectors to output the ASDF files
for wfi_num in range(18):
    detector = wfi_num + 1
    # Update the detector number in the metadata
    tmp.meta_photom.instrument_detector = f'WFI{detector:02d}'

    
    # Build the photom dictionary
    rfp_photom.make_photom()
    
    # Generate the CRDS-compliant reference file
    rfp_photom.outfile = f'roman_photom_file_wfi{detector:02}.asdf'
    rfp_photom.generate_outfile()

Let's check one of the output files to verify if everything looks good. 

In [ ]:
import asdf

# Turn off the lazy_load to unload the wavelength and effective area arrays
with asdf.open(rfp_photom.outfile, lazy_load=False) as af:
    photom = af["roman"]

-----

This notebook was written by Eunkyu Han on June 25, 2026.